# Step 2 - Abstraction #

1. From local 3D points
2. From 2D RGB image
3. From All 3D points
4. Higher Level Structural Analysis

All these abstractions might not suite every capture, need designers' insights to select the best ones.

Also need to work on user interaction and visualization.

In [1]:
from pathlib import Path
import sys
from pipeline import abstraction
import open3d as o3d

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [2]:
VIGNETTE_NAME = "book"
VIGNETTE_PATH = project_root / "test_data" / VIGNETTE_NAME

rgb_path = VIGNETTE_PATH / "rgb.png"
results_path = VIGNETTE_PATH / "results"

## 2.1 Abstraction from Local 3D Points ##

Values stored in each point as per-point attribute.

- curvature
- edgeness
- anisotropy
- planarity
- sphericity
- flow_vectors

In [ ]:
# Load vignette
processed_vignette_path = results_path / "processed_vignette.npz"
from pipeline.vignette_data import ProcessedVignette
processed_vignette = ProcessedVignette.load(processed_vignette_path)

In [ ]:
# run code from abstraction
abstraction.analyze_local_features(
    processed_vignette, 
    search_radius=0.03, 
    max_neighbors=30,
    auto_save=True
)

### Visualize Points ###

V1 - Visualize all points

In [ ]:
# Visualize features with points

# 'curvature', 'edgeness', 'anisotropy', 'planarity', 'sphericity'
color_mode = 'planarity' 

print(f"Visualizing {color_mode}...")
o3d.visualization.draw_geometries([processed_vignette.to_open3d(color_mode=color_mode)])

V2 - Visualize points under threshold

In [ ]:
# Examples - points closer to edges
pcd = processed_vignette.to_open3d_threshold("edgeness", 0.6, direction="above", inclusive=True)
o3d.visualization.draw_geometries([pcd])

In [ ]:
# Points that are more planner
pcd = processed_vignette.to_open3d_threshold("planarity", 0.5, direction="above", inclusive=True)
o3d.visualization.draw_geometries([pcd])

V3 - Visualize flow vectors

In [ ]:
from pipeline.preview_helper import visualize_vector_attribute

visualize_vector_attribute(processed_vignette, 'flow_vectors', step=10, scale=0.1)

## 2.2 Abstraction from 2D RGB Image ##

- Convert rgb image into abstracted images, apply to 3D points for per-point attribute
    - Canny edges with dilation
    - High-frequency detail
    - Stylized image with K-means color quantization

- Global values as metadata
    - Color palette

In [3]:
# Load vignette
processed_vignette_path = results_path / "processed_vignette.npz"
from pipeline.vignette_data import ProcessedVignette
processed_vignette = ProcessedVignette.load(processed_vignette_path)

Set/updated per-point attribute: 'confidence'.
Set/updated per-point attribute: 'component_id'.
Set/updated per-point attribute: '2d_edges'.
Set/updated per-point attribute: '2d_texture_detail'.
Set/updated per-point attribute: 'stylized_colors'.
Loaded processed vignette from: /Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/test_data/tree_bark/results/processed_vignette.npz


In [4]:
# Input: rgb, mask, vignette
rgb_path = VIGNETTE_PATH / "rgb.png"
mask_path = results_path / "mask.png"

# Output
output_dir = VIGNETTE_PATH / "results" / "feature_maps"
output_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
print("--- Generating 2D Feature Maps ---")
edge_map_path = abstraction.generate_edge_map(
    rgb_path, 
    mask_path, 
    str(output_dir / "edges.png"), 
    canny_thresh1=300, canny_thresh2=800, 
    dilation_kernel_size=6, dilation_iterations=1
)
detail_map_path = abstraction.generate_detail_map(rgb_path, mask_path, str(output_dir / "detail.png"))
stylized_map_path = abstraction.generate_stylized_image(rgb_path, mask_path, str(output_dir / "stylized_k4.png"), k=4)


In [ ]:
print("--- Applying Feature Maps to Vignette ---")
processed_vignette = abstraction.apply_feature_map_to_vignette(processed_vignette, edge_map_path, '2d_edges')
processed_vignette = abstraction.apply_feature_map_to_vignette(processed_vignette, detail_map_path, '2d_texture_detail')
processed_vignette = abstraction.apply_feature_map_to_vignette(processed_vignette, stylized_map_path, 'stylized_colors')
processed_vignette.save()

In [ ]:
print("Visualizing the new 'stylized_colors' attribute...")
pcd_stylized = processed_vignette.to_open3d(color_mode='rgb') # Start with original colors
pcd_stylized.colors = o3d.utility.Vector3dVector(processed_vignette.stylized_colors)
o3d.visualization.draw_geometries([pcd_stylized])


Visualizing the new 'stylized_colors' attribute...
Generating Open3D point cloud with 'rgb' colors...


In [ ]:
print("--- Extracting Color Palette ---")
processed_vignette = abstraction.extract_color_palette(
    processed_vignette,
    rgb_path,
    mask_path,
    k=8,
    output_dir=str(output_dir),
    auto_save=True
)


--- Extracting Color Palette ---
Extracting 8-color palette...
Set/updated metadata property: 'color_palette'.
Saved processed vignette to: /Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/test_data/tree_bark/results/processed_vignette.npz
   - Saved palette visualization to /Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/test_data/tree_bark/results/feature_maps/palette.png


In [ ]:
# print palette
palette = processed_vignette.metadata.get('color_palette')
if palette:
    print("Extracted Palette:")
    print(f"  Number of colors (k): {palette['k']}")
    for color, prop in zip(palette['colors_rgb_255'], palette['proportions']):
        print(f"  - Color (RGB): {color}, Proportion: {prop:.2%}")


Extracted Palette:
  Number of colors (k): 8
  - Color (RGB): [103, 114, 73], Proportion: 17.90%
  - Color (RGB): [133, 141, 111], Proportion: 16.32%
  - Color (RGB): [71, 81, 42], Proportion: 15.61%
  - Color (RGB): [166, 168, 152], Proportion: 13.06%
  - Color (RGB): [201, 204, 184], Proportion: 12.88%
  - Color (RGB): [234, 235, 228], Proportion: 9.09%
  - Color (RGB): [34, 42, 15], Proportion: 8.46%
  - Color (RGB): [173, 197, 97], Proportion: 6.68%


## 2.3 Abstraction from All 3D Points ##

In [3]:
# Load preprocessed vignette for tests
processed_vignette_path = results_path / "processed_vignette.npz"
from pipeline.vignette_data import ProcessedVignette
processed_vignette = ProcessedVignette.load(processed_vignette_path)

Set/updated per-point attribute: 'confidence'.
Set/updated per-point attribute: 'component_id'.
Set/updated per-point attribute: 'curvature'.
Set/updated per-point attribute: 'edgeness'.
Set/updated per-point attribute: 'anisotropy'.
Set/updated per-point attribute: 'planarity'.
Set/updated per-point attribute: 'sphericity'.
Set/updated per-point attribute: 'flow_vectors'.
Set/updated per-point attribute: '2d_edges'.
Set/updated per-point attribute: '2d_texture_detail'.
Set/updated per-point attribute: 'stylized_colors'.
Set/updated per-point attribute: 'plane_id'.
Set/updated per-point attribute: 'cylinder_id'.
Set/updated per-point attribute: 'sphere_id'.
Set/updated per-point attribute: 'cuboid_id'.
Set/updated per-point attribute: 'best_fit_id'.
Loaded processed vignette from: /Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/test_data/book/results/processed_vignette.npz


### Structural Properties with PCA ###

Added these values to metadata

- `structural_properties`
    - `type`: global or component
    - `component_id`: if component
    - `centroid`: xyz coordinate of the center of points
    - `axes`: 3x3 matrix, the 3 principal axes (eigenvectors) perpendicular to each other
    - `variances`: 3 floats, eigenvalues for each axis
    - `linearity`: how line-like the shape is. 1.0 is long and thin
    - `planarity`: how plane-like the shape is. 1.0 is very flat
    - `sphericity`: how sphere-like the shape is. 1.0 means no dominant direction
    - `anisotropy`: how directional the shape is. 1.0 means has clear orientation

In [4]:
# PCA
abstraction.analyze_structural_properties(
    processed_vignette, auto_save=True
)

Analyzing structural properties with PCA...
Added new abstraction of type 'structural_properties'.
   - Global properties: L=0.53, P=0.46, S=0.00
   - Skipping per-component analysis (1 component found).
Saved processed vignette to: /Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/test_data/book/results/processed_vignette.npz
Finished structural property analysis.


In [6]:
# Visualize PCA Axes
from pipeline.preview_helper import visualize_structural_properties
visualize_structural_properties(
    processed_vignette,
    axis_scale_factor=1.5
)

Generating Open3D point cloud with 'component_id' colors...
Generating PCA axis visualizations...
 - Added GLOBAL axes.

Launching Open3D visualizer...


### Fit Primitives with pyransac3d ###

Added these values to metadata
- abstractions
    - planes
    - cylinders
    - spheres
    - cuboids

Common attributes for each primitive
- `type_id`: integer id
- `source_attribute`: For instance  `{'component_id': 2}`
- `point_count`: number of inlier points
- `fit_error`: average distance in meters from points to the ideal mathematical shape
- `obb_center`: xyz center of the Oriented Bounding Box
- `obb_rotation`: 3x3 rotation matrix for the orientation of OBB
- `obb_extent`: length, width, height of the OBB along its local axes

For planes
- `equation`: [a, b, c, d] for plane ax + by + cz + d = 0

For cylinders
- `center`: xyz center
- `axis`: dxdydz for the direction vector of the central axis
- `radius`: radius of cylinder in meters

For spheres
- `center`: xyz center
- `radius`: radius of sphere in meters

#### Plane ####

In [4]:
# planes
abstraction.extract_primitives_by_component(
    vignette=processed_vignette,
    primitive_type='plane',
    distance_threshold=0.02,
    min_points=100,
    plane_method='ransac'
)

Extracting planes grouped by 'component_id'...
   - Using plane fitting method: 'ransac'
Cleared 2 abstractions of type 'planes'.
   - No components found or only one component. Falling back to entire point cloud.
Set/updated per-point attribute: '__temp_group_id'.
   - Processing subset where '__temp_group_id' == 0...
     - Found 1 primitive(s).
   - Cleaned up temporary attribute '__temp_group_id'.

   - Added a total of 1 new abstractions of type 'planes'.
Set/updated per-point attribute: 'plane_id'.
Saved processed vignette to: /Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/test_data/book/results/processed_vignette.npz
Finished extracting dominant planes.


In [13]:
from pipeline.preview_helper import visualize_primitives
visualize_primitives(processed_vignette, 'plane')

--- Visualizing Planes ---
Generating Open3D point cloud with 'plane_id' colors...
   - Coloring points by 'plane_id'.
   - Found 1 primitives to render as meshes.

Launching Open3D visualizer...


#### Cylinder ####

In [5]:
# cylinders
abstraction.extract_primitives_by_component(
    vignette=processed_vignette,
    primitive_type='cylinder',
    distance_threshold=0.1,
    min_points=100
)

Extracting cylinders grouped by 'component_id'...
Cleared 1 abstractions of type 'cylinders'.
   - No components found or only one component. Falling back to entire point cloud.
Set/updated per-point attribute: '__temp_group_id'.
   - Processing subset where '__temp_group_id' == 0...


/Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/venv/lib/python3.12/site-packages/pyransac3d/cylinder.py:77: RuntimeWarning: divide by zero encountered in scalar divide
  ma = (P_rot[1, 1] - P_rot[0, 1]) / (P_rot[1, 0] - P_rot[0, 0])
/Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/venv/lib/python3.12/site-packages/pyransac3d/cylinder.py:85: RuntimeWarning: invalid value encountered in scalar divide
  p_center_x = (
/Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/venv/lib/python3.12/site-packages/pyransac3d/cylinder.py:86: RuntimeWarning: invalid value encountered in scalar subtract
  ma * mb * (P_rot[0, 1] - P_rot[2, 1])
/Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/venv/lib/python3.12/site-packages/pyransac3d/cylinder.py:78: RuntimeWarning: divide by zero encountered in scalar divide
  mb = (P_rot[2, 1] - P_rot[1, 1]) / (P_rot[2, 0] - P_rot[1, 0])


     - Found 1 primitive(s).
   - Cleaned up temporary attribute '__temp_group_id'.

   - Added a total of 1 new abstractions of type 'cylinders'.
Set/updated per-point attribute: 'cylinder_id'.
Saved processed vignette to: /Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/test_data/book/results/processed_vignette.npz
Finished extracting dominant cylinders.


In [15]:
from pipeline.preview_helper import visualize_primitives
visualize_primitives(processed_vignette, 'cylinder')

--- Visualizing Cylinders ---
Generating Open3D point cloud with 'cylinder_id' colors...
   - Coloring points by 'cylinder_id'.
   - Found 1 primitives to render as meshes.

Launching Open3D visualizer...


#### Sphere ####

In [6]:
# spheres
abstraction.extract_primitives_by_component(
    vignette=processed_vignette,
    primitive_type='sphere',
    distance_threshold=0.1,
    min_points=100
)

Extracting spheres grouped by 'component_id'...
Cleared 1 abstractions of type 'spheres'.
   - No components found or only one component. Falling back to entire point cloud.
Set/updated per-point attribute: '__temp_group_id'.
   - Processing subset where '__temp_group_id' == 0...
     - Found 1 primitive(s).
   - Cleaned up temporary attribute '__temp_group_id'.

   - Added a total of 1 new abstractions of type 'spheres'.
Set/updated per-point attribute: 'sphere_id'.
Saved processed vignette to: /Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/test_data/book/results/processed_vignette.npz
Finished extracting dominant spheres.


In [17]:
from pipeline.preview_helper import visualize_primitives
visualize_primitives(processed_vignette, 'sphere')

--- Visualizing Spheres ---
Generating Open3D point cloud with 'sphere_id' colors...
   - Coloring points by 'sphere_id'.
   - Found 1 primitives to render as meshes.

Launching Open3D visualizer...


#### Cuboid ####

In [7]:
# cuboid
abstraction.extract_primitives_by_component(
    vignette=processed_vignette,
    primitive_type='cuboid',
    distance_threshold=0.01,
    min_points=100
)

Extracting cuboids grouped by 'component_id'...
Cleared 2 abstractions of type 'cuboids'.
   - No components found or only one component. Falling back to entire point cloud.
Set/updated per-point attribute: '__temp_group_id'.
   - Processing subset where '__temp_group_id' == 0...
     - Found 1 primitive(s).
   - Cleaned up temporary attribute '__temp_group_id'.

   - Added a total of 1 new abstractions of type 'cuboids'.
Set/updated per-point attribute: 'cuboid_id'.
Saved processed vignette to: /Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/test_data/book/results/processed_vignette.npz
Finished extracting dominant cuboids.


In [19]:
from pipeline.preview_helper import visualize_primitives
visualize_primitives(processed_vignette, 'cuboid')

--- Visualizing Cuboids ---
Generating Open3D point cloud with 'cuboid_id' colors...
   - Coloring points by 'cuboid_id'.
   - Found 2 primitives to render as meshes.

Launching Open3D visualizer...


### Fit Planes with Region Growing Segmentation ###

In [8]:
# Need to explore the effects of all parameters
abstraction.extract_primitives_by_component(
    vignette=processed_vignette,
    primitive_type='plane',
    distance_threshold=0.001, # Changing subdivision threashold
    min_points=60,
    plane_method='region_growing',
    normal_angle_tolerance_deg=5, # the smaller the stricter
    min_samples_normals_ratio=0.001, # minimum size of points to be considered a valid plane segmentation
    noise_strategy_layer='merge'
)
# noise_strategy_layer
# "force_assign" (merge all to plane) | "merge" (merge to plane within distance) | "new_plane" (create a new plane for noise) | "keep" (keep as noise)

Extracting planes grouped by 'component_id'...
   - Using plane fitting method: 'region_growing'
Cleared 1 abstractions of type 'planes'.
   - No components found or only one component. Falling back to entire point cloud.
Set/updated per-point attribute: '__temp_group_id'.
   - Processing subset where '__temp_group_id' == 0...
     [DEBUG] --- Region Growing (planes) on 10616 points ---
     [DEBUG] [1/5] Estimating normals...
     [DEBUG]        - normal_search_radius: 0.005000
     [DEBUG]        - normal_max_nn: 30
     [DEBUG] [2/5] Canonicalizing normal directions...
     [DEBUG]        - Flipped 77 normals.
     [DEBUG] [3/5] Orientation clustering (DBSCAN on unit normals)...
     [DEBUG]        - angle_tol_deg=5.000 -> eps_normals=0.087239
     [DEBUG]        - min_samples_normals=60
     [DEBUG]        - Unique orient labels: [np.int64(-1), np.int64(0), np.int64(1)] (noise=-1)

     [DEBUG] -> Orientation group #0 | size=9042
     [DEBUG]      - Projection normal (unit): [ 0.57

In [15]:
from pipeline.preview_helper import visualize_primitives
visualize_primitives(processed_vignette, 'plane')

--- Visualizing Planes ---
Generating Open3D point cloud with 'plane_id' colors...
   - Coloring points by 'plane_id'.
   - Found 2 primitives to render as meshes.

Launching Open3D visualizer...


### Best Primitives ###

score = point_count / (fit_error + epsilon)

Added these values to metadata
- best_fit_composition
    - best_fit_id
    - score
    - All the original keys (type, point_count, etc.)

In [9]:
abstraction.compose_best_fit_abstraction(
    processed_vignette,
    min_coverage_ratio=0.005, # primitive coverage above this are considered
    score_alpha=2.0          # higher means accuracy > coverage
)

Composing best-fit abstraction...
   - Parameters: min_coverage_ratio=0.005, score_alpha=2.0
   - Starting greedy selection. A primitive must explain at least 53 new points.

     + Evaluating Candidate #1: planes (Source ID: 1)
       - Stats: Point Count=8932, Fit Error=0.0010
       - Calculated Score: 8557497093.48
       - It has 8932 total points. Of those, 8932 are currently unexplained.
       - ACCEPTED: 8932 >= threshold 53.

     + Evaluating Candidate #2: planes (Source ID: 2)
       - Stats: Point Count=447, Fit Error=0.0006
       - Calculated Score: 1268949705.50
       - It has 447 total points. Of those, 447 are currently unexplained.
       - ACCEPTED: 447 >= threshold 53.

     + Evaluating Candidate #3: spheres (Source ID: 1)
       - Stats: Point Count=10616, Fit Error=0.0043
       - Calculated Score: 585818255.01
       - It has 10616 total points. Of those, 1237 are currently unexplained.
       - ACCEPTED: 1237 >= threshold 53.

     + Evaluating Candidate #4: 

### Symmetry ###

Added these values to metadata
- `symmetries`
    - `type`: reflectional
    - `score`: mean_inlier_error / inlier_ratio. Lower is better
    - `mean_inlier_error`: average distance between reflected points and their actual partners
    - `plane_point`: centroid of object
    - `plane_normal`: orientation of the symmetry plane

In [10]:
abstraction.analyze_global_symmetry(
    vignette=processed_vignette, 
    match_threshold=0.03, # search distance to find pairs. Smaller for more precise points
    min_inlier_ratio=0.5, # How many points need to have a symmetric partner
    auto_save=True
)

Analyzing global symmetry (robust method)...
Cleared 1 abstractions of type 'symmetries'.
   [DEBUG] Initial point count: 10616
   [DEBUG] 'component_id' attribute found. Attempting to filter noise...
   [DEBUG] WARNING: Found only 0 non-noise points, which is below the threshold.
   [DEBUG] This likely means component segmentation failed. Using ALL points as a fallback.
   --- Testing Reflectional Symmetry ---
   -> Testing plane normal to 'Primary' axis...
      [DEBUG] Potential inliers found (distance < 0.03m): 10570 / 10616
      [DEBUG] Consistent inliers found (passed backward check): 5869 / 10570
      [DEBUG] Result: Score=0.0023, Inlier Ratio=55.28%, Mean Error=0.0013m
   -> Testing plane normal to 'Secondary' axis...
      [DEBUG] Potential inliers found (distance < 0.03m): 10604 / 10616
      [DEBUG] Consistent inliers found (passed backward check): 5566 / 10604
      [DEBUG] Result: Score=0.0025, Inlier Ratio=52.43%, Mean Error=0.0013m
   -> Testing plane normal to 'Tertia

## 2.4 Higher Level Structural Analysis ##

### Inter-Primitive Relationships ###

Values added to metadata
-`abstractions` - `primitive_relations`
    - `type`: parallel, perpendicular, or co-planar
    - `primitives`: [type_id1, type_id2]

For plane-plane relationships
- `angle_diff_deg`: deviation from perfect relationship: parallel - angle from 0, perpendicular - angle from 90
- `distance_m`: for parallel and co-planar

In [ ]:
abstraction.analyze_primitive_relations(
    vignette=processed_vignette,
    angle_tolerance_deg=10.0, # Degree
    distance_tolerance_m=0.002, # Distance in meter
    auto_save=True
)

Analyzing primitive relationships...
   [DEBUG] Angle tolerance: 10.0°
   [DEBUG] Parallel check: dot product must be > 0.9848
   [DEBUG] Perpendicular check: dot product must be < 0.1736

   --- Analyzing 2 Plane-Plane pairs ---

   -> Comparing plane_1 and plane_2...
      [DEBUG] Normalized dot product: 0.1464
      [DEBUG] Test PASS: 0.1464 < 0.1736 (Perpendicular)
      -> SUCCESS: Found 'PERPENDICULAR' relationship.
Added new abstraction of type 'primitive_relations'.

   - Skipping cylinder-cylinder analysis (less than 2 cylinders found).
Saved processed vignette to: /Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/test_data/book/results/processed_vignette.npz

Finished analyzing primitive relationships.


### Something Else ###

- Parallel patterns
- Perpendicular intersection lines, corners
- Dominant void - get a point cloud of the empty space